# Opportunity Agent

Runs the full pipeline: **Goal -> Plan -> Retrieve -> Analyze -> Decide -> Recommend -> Validate**. Every stage is a separate, logged tool call - the model plans before investigating, analyzes the evidence before deciding, and a second pass validates the draft recommendations against that evidence before anything is saved.

In [1]:
import sys
from pathlib import Path

import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

def _locate_and_import_agent_core():
    here = Path.cwd()
    candidates = [here, here / "agents", here.parent / "agents", here.parent]
    for c in candidates:
        if (c / "agent_core.py").exists():
            sys.path.insert(0, str(c))
            import agent_core as agent_core_module
            return agent_core_module
    raise FileNotFoundError(
        "Could not locate agent_core.py. Looked in: "
        + ", ".join(str(c) for c in candidates)
        + f". Current working directory: {here}"
    )


core = _locate_and_import_agent_core()

DATA_DIR = core.find_data_dir()
CHUNKS_PATH   = DATA_DIR / "chunked_data.json"
INDEX_PATH    = DATA_DIR / "sap_intelligence.index"
OUT_PATH      = DATA_DIR / "opportunities.json"
PIPELINE_PATH = DATA_DIR / "opportunities_pipeline.json"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

GOAL = (
    "Identify 3 to 5 concrete current business opportunities for SAP - "
    "new markets, partnerships, AI products, or revenue growth - using "
    "the indexed SAP news corpus as evidence."
)

print("data dir:", DATA_DIR)


data dir: /home/jovyan/vault/Testing/Testing/notebook/data


In [2]:
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
embed_model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)

core.warmup()


chunks loaded: 1400
vectors in index: 1400
warming up model...


warmup done in 28.3 sec


In [3]:
# ----------------------------------------------------------------
# Retrieve-stage tools: the agent decides the queries and how many
# times to call these. read_previous_findings gives it memory of
# its last run.
# ----------------------------------------------------------------
def retrieve_news(query, k=6):
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index.search(q_vec, k)
    return chunks_df.iloc[idx[0]]["text"].tolist()


def read_previous_findings(_args=None):
    previous = core.load_previous_output(OUT_PATH)
    if not previous:
        return "No previous findings on file - this looks like the first run."
    return previous


RETRIEVAL_TOOL_HANDLERS = {
    "retrieve_news": lambda args: retrieve_news(args.get("query", ""), int(args.get("k", 6) or 6)),
    "read_previous_findings": lambda args: read_previous_findings(args),
}

RETRIEVAL_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_news",
            "description": (
                "Semantic search over the indexed SAP news corpus. Call this "
                "with your own queries to investigate different angles - new "
                "markets, partnerships, AI products, revenue growth, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural language search query"},
                    "k": {"type": "integer", "description": "Number of chunks to retrieve (default 6)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_previous_findings",
            "description": (
                "Read the opportunities you identified in the previous run, "
                "if any, so you can check what is still supported by current "
                "evidence."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
]


In [4]:
# ----------------------------------------------------------------
# Decide and Validate tool schemas (Analyze reuses the generic
# helper in agent_core, since "key observations + candidate items"
# is the same shape for every domain).
# ----------------------------------------------------------------
DECIDE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "finalize_opportunities",
        "description": (
            "Call this exactly once to submit your final list of business "
            "opportunities."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "opportunities": {
                    "type": "array",
                    "description": "3 to 5 concrete business opportunities",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "impact_level": {"type": "string", "description": "High, Medium, or Low"},
                            "evidence": {"type": "string"},
                            "confidence_score": {"type": "integer", "description": "0 to 100"},
                        },
                        "required": ["title", "impact_level", "evidence", "confidence_score"],
                    },
                }
            },
            "required": ["opportunities"],
        },
    },
}

VALIDATE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "submit_validated_opportunities",
        "description": (
            "Call this exactly once to submit your evidence-checked, "
            "final list of opportunities."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "validated_opportunities": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "impact_level": {"type": "string"},
                            "evidence": {"type": "string"},
                            "confidence_score": {"type": "integer"},
                            "validation_status": {"type": "string", "description": "Supported, Revised, or Unsupported"},
                            "validation_notes": {"type": "string"},
                        },
                        "required": ["title", "impact_level", "evidence", "confidence_score", "validation_status", "validation_notes"],
                    },
                }
            },
            "required": ["validated_opportunities"],
        },
    },
}


In [5]:
validated_opportunities, pipeline = core.run_full_pipeline(
    goal_description=GOAL,
    retrieval_tool_schemas=RETRIEVAL_TOOL_SCHEMAS,
    retrieval_tool_handlers=RETRIEVAL_TOOL_HANDLERS,
    analyze_tool_schema=core.make_analyze_tool_schema("opportunities"),
    decide_tool_schema=DECIDE_TOOL_SCHEMA,
    decide_tool_name="finalize_opportunities",
    decide_items_key="opportunities",
    validate_tool_schema=VALIDATE_TOOL_SCHEMA,
    validate_tool_name="submit_validated_opportunities",
    validate_items_key="validated_opportunities",
)

print()
print("=" * 70)
print(len(validated_opportunities), "validated opportunities:")
for opp in validated_opportunities:
    print("-", opp.get("title"), "|", opp.get("impact_level"), "|", opp.get("validation_status"))



STAGE 1: PLAN


PLAN: {
  "goal": "Identify potential business opportunities for SAP in new markets, partnerships, AI products, or revenue growth by leveraging the indexed SAP news corpus.",
  "planned_steps": [
    "Search the SAP news corpus for mentions of new market expansions and partnerships",
    "Analyze recent announcements and press releases from SAP on AI product development and innovation",
    "Examine industry reports and research studies citing SAP as a leader in specific markets or revenue growth opportunities"
  ]
}

STAGE 2: RETRIEVE


  [retrieve step 1] called `retrieve_news` with {"k": "6", "query": "new market expansions and partnerships"}


  [retrieve step 2] called `retrieve_news` with {"k": "6", "query": "SAP news on AI product development and innovation"}


  [retrieve step 3] called `retrieve_news` with {"k": "6", "query": "SAP as a leader in specific markets or revenue growth opportunities"}


  [retrieve step 4] no tool call - treating retrieval as complete.

STAGE 3: ANALYZE


ANALYSIS: {
  "candidate_items": [
    "New Market Expansion and Partnership Opportunities",
    "AI Product Development and Innovation Opportunities",
    "Revenue Growth Opportunities in SAP-led Markets"
  ],
  "key_observations": [
    "SAP is expanding its economic model for partners, creating a more predictable revenue model.",
    "The company's AI-driven solutions are expected to improve customer experience and drive business outcomes.",
    "Partners can build industry solutions and implementation packages with SAP's Autonomous CX.",
    "SAP provides a strong starting point for embedded assistants and standard integrations, with prebuilt industry scenarios.",
    "The company's updated portfolio addresses the need for quick adaptation to market changes, AI-driven innovations, and business continuity."
  ]
}

STAGE 4: DECIDE + RECOMMEND


DRAFT DECISION: {
  "opportunities": "[{\"title\": \"New Market Expansion and Partnership Opportunities\", \"impact_level\": \"High\", \"evidence\": \"SAP is expanding its economic model for partners, creating a more predictable revenue model.\", \"confidence_score\": 90}, {\"title\": \"AI Product Development and Innovation Opportunities\", \"impact_level\": \"Medium\", \"evidence\": \"The company's AI-driven solutions are expected to improve customer experience and drive business outcomes.\", \"confidence_score\": 80}, {\"title\": \"Revenue Growth Opportunities in SAP-led Markets\", \"impact_level\": \"High\", \"evidence\": \"Partners can build industry solutions and implementation packages with SAP's Autonomous CX.\", \"confidence_score\": 95}]"
}

STAGE 5: VALIDATE


VALIDATED: {
  "validated_opportunities": "[{\"title\": \"New Market Expansion and Partnership Opportunities\", \"impact_level\": \"High\", \"evidence\": \"SAP is expanding its economic model for partners, creating a more predictable revenue model.\", \"confidence_score\": 90}, {\"title\": \"AI Product Development and Innovation Opportunities\", \"impact_level\": \"Medium\", \"evidence\": \"The company's AI-driven solutions are expected to improve customer experience and drive business outcomes.\", \"confidence_score\": 80}, {\"title\": \"Revenue Growth Opportunities in SAP-led Markets\", \"impact_level\": \"High\", \"evidence\": \"Partners can build industry solutions and implementation packages with SAP's Autonomous CX.\", \"confidence_score\": 95}]"
}

3 validated opportunities:
- New Market Expansion and Partnership Opportunities | High | None
- AI Product Development and Innovation Opportunities | Medium | None
- Revenue Growth Opportunities in SAP-led Markets | High | None


In [6]:
core.save_json(OUT_PATH, validated_opportunities)
core.save_json(PIPELINE_PATH, pipeline)


saved to /home/jovyan/vault/Testing/Testing/notebook/data/opportunities.json
saved to /home/jovyan/vault/Testing/Testing/notebook/data/opportunities_pipeline.json
